# 04 — Model Explainability

**Hospital Readmission Risk Predictor**

A model that cannot be explained cannot be trusted in a clinical setting. This notebook uses
SHAP (SHapley Additive exPlanations) to understand *why* the model makes specific predictions,
both at a global level (which features matter most?) and at a local level (why was *this*
patient flagged?).

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.data_loader import load_and_prepare
from src.leakage_detection import remove_leakage_rows
from src.feature_engineering import engineer_features
from src.preprocessing import resolve_feature_lists, split_X_y
from src.explainability import (
    compute_shap_values, plot_shap_summary,
    analyse_top_features, generate_local_explanations
)
from src.utils import set_plot_style, load_pipeline, set_seed
from src.config import RANDOM_SEED, TEST_SIZE

from sklearn.model_selection import train_test_split

set_plot_style()
set_seed()
print('Setup complete ✓')

In [ ]:
# Load data and pipeline
df = load_and_prepare()
df = remove_leakage_rows(df)
df = engineer_features(df)

X, y = split_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)

pipeline = load_pipeline()
print(f'Pipeline loaded. Test set: {len(X_test):,} samples')

## 1. Global Explainability — SHAP Summary

The beeswarm plot shows how each feature contributes to predictions across the entire test set.
Each dot represents one patient. The x-axis shows the SHAP value (impact on model output),
and colour indicates the feature value (red = high, blue = low).

In [ ]:
shap_values, X_transformed, feature_names = compute_shap_values(pipeline, X_test)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
plot_shap_summary(shap_values, X_transformed, feature_names)

## 2. Top Features Analysis

Let's examine the top 20 features ranked by mean absolute SHAP value and discuss their
clinical significance.

In [ ]:
top_features = analyse_top_features(shap_values, feature_names, top_n=20)
display(top_features)

### Clinical Interpretation of Top Features

The features driving readmission risk predictions align well with clinical knowledge:

1. **number_inpatient** — Prior hospitalisations are the single strongest predictor. Patients
   with frequent prior admissions are likely managing complex chronic conditions and have
   established patterns of acute healthcare utilisation.

2. **number_diagnoses** — A higher number of diagnoses indicates greater clinical complexity
   (multimorbidity). These patients require coordinated care across multiple specialists,
   which increases the risk of post-discharge complications.

3. **number_emergency** — Emergency department visits reflect both disease severity and
   potential gaps in outpatient follow-up. Frequent ED users often lack reliable primary care.

4. **prior_visits_total** — Our engineered composite utilisation score captures the overall
   pattern of healthcare consumption.

5. **Discharge disposition** — Where patients go after discharge (home, SNF, another facility)
   reflects their functional status and available support systems.

6. **num_medications** — Polypharmacy is a well-established readmission risk factor. More
   medications mean more drug interactions, more side effects, and more opportunities for
   non-adherence.

7. **Diagnosis categories** — Circulatory conditions (heart failure, stroke) and diabetes
   as a primary diagnosis are known high-risk categories.

These findings give clinicians confidence that the model is learning *real* clinical signals
rather than data artefacts.

## 3. Local Explanations

Global explanations tell us *what* the model looks at; local explanations tell us *why* a
specific prediction was made. We examine three cases:

1. **True Positive** — A patient correctly flagged for readmission
2. **False Positive** — A patient incorrectly flagged (alarm fatigue risk)
3. **False Negative** — A readmitted patient the model missed (the costly error)

In [ ]:
# Generate predictions at our optimised threshold
y_prob = pipeline.predict_proba(X_test)[:, 1]
threshold = 0.3  # Will be refined in practice
y_pred = (y_prob >= threshold).astype(int)

paths = generate_local_explanations(
    pipeline, X_test, y_test, y_pred,
    shap_values, X_transformed, feature_names
)

for label, path in paths.items():
    print(f'{label.upper()} plot saved: {path}')

### True Positive — Why the Model Succeeded

In a correctly-flagged readmission, we typically see high SHAP values for clinical risk factors
like prior inpatient visits, multiple diagnoses, and medication complexity. The model correctly
identified the constellation of risk factors that make readmission likely.

### False Positive — Why the Model Over-predicted

False positives often occur when a patient has some risk factors (e.g. older age, several
medications) but lacked the specific combination that actually leads to readmission. These
cases are tolerable in a clinical programme — the patient receives extra follow-up that
does no harm.

### False Negative — Why the Model Missed It

Missed readmissions are the most clinically consequential errors. They often involve patients
whose observable features at discharge looked relatively benign — perhaps a patient with few
prior visits who developed a post-discharge complication. These cases highlight the model's
limitations: it can only predict based on available data, not unforeseen events.

## Summary

SHAP analysis confirms that the model:

- Relies on clinically sensible features (prior visits, complexity, medications)
- Does NOT rely on demographic attributes as primary drivers (supporting fairness)
- Can provide per-patient explanations suitable for clinical decision support

These explanations are critical for regulatory compliance, clinical trust, and ultimately
patient safety.